<a href="https://colab.research.google.com/github/elenamaggiore/Thesis_Elena_Maggiore/blob/preprocessing/PREPROCESSING_masking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PREPROCESSING:
## MASKING

This notebook will address the datasets' problems and prepare them for their use in the analysis.

* First, a preprocessing through REGEX will be applied.

## GENERAL:

Libraries, mounting Google Drive, and reading files.

In [ ]:
%pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [ ]:
from __future__ import annotations
import sys
import subprocess

try:
    import cudf as pd
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "numba", "cudf-cu12", "--extra-index-url=https://pypi.nvidia.com"], stdout=subprocess.DEVNULL)
    try:
        import cudf as pd
    except Exception:
        import pandas as pd

from google.colab import drive
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import gc
import os
import ast, gc, hashlib, html, itertools, json, math, random, re

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## MASKING FILTER

This section will recognise the following elements from users' comments:
* Mention of MBTI types, case insensitive, considering both singular and plural.
* Mention of Cognitive Functions.
* Specific words and expressions like "personality test", "myers - briggs", the Big Five categories, and so on.
* URLs.
* Comments that are generated by bots.
* Comments that are deleted.

In [ ]:
TYPES = ["".join(t) for t in itertools.product("EI", "NS", "TF", "JP")]

TYPE_RE = re.compile(
    r"\b[EI][NS][TF][JP](?:s|'s)?\b", re.I
)

FUNC_RE = re.compile(
    r"(?<![\w-])(?:[NSTF][ie])+(?!\w)"
)

PLACEHOLDER_RE = re.compile(
    r"\b[A-Z]+\_MENTION\b"
)

VOCAB_RE = re.compile(
    r"\b(?:mbti|myers[- ]?briggs|enneagram|socionics|typology|cognitive functions?|16 ?personalit(?:y|ies)|"
    r"personality (?:test|type|quiz)s?|big ?(?:five|5)|neo[- ]?(?:pi|ffi)|ipip|truity|hexaco|"
    r"openness|conscientiousness|extraversion|extroversion|agreeableness|neuroticism)\b",
    re.I
)

LEAK_PATTERNS = (
    TYPE_RE,
    FUNC_RE,
    PLACEHOLDER_RE,
    VOCAB_RE
)

URL_RE = re.compile(
    r"https?://\S+|www\.\S+",
    re.I
)

BOT_RE = re.compile(
    r"i am a bot|this action was performed automatically|bleep,? bloop",
    re.I
)

DELETED = {"[deleted]", "[removed]"}

These functions will be used to determine if a specific comment is a source of data leak, labelling it _ok_, _leaky_, _short_, _with placeholder_, or _bad_. Then, bad and short comments will be removed, leaky comments will be fixed with placeholders and kept together with the rest.

In [ ]:
def has_leak(text):
    return any(p.search(text) for p in LEAK_PATTERNS)

def leak_hits(text):
    return [
        m.group(0)
        for p in LEAK_PATTERNS
        for m in p.finditer(text)
    ]

def parse_list(x):
    if isinstance(x, list):
        return x

    if not isinstance(x, str) or not x:
        return []

    try:
        return json.loads(x)
    except ValueError:
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []


def author_hash(a):
  return hashlib.sha256(str(a).encode()).hexdigest()[:10]



def clean_comment(raw):
    if not isinstance(raw, str):
        return None

    t = html.unescape(raw).strip()

    if t.lower() in DELETED or BOT_RE.search(t):
        return None

    # Rimuove le righe citate
    t = " ".join(
        line for line in t.splitlines()
        if not line.lstrip().startswith(">")
    )

    # Rimuove gli URL e normalizza gli spazi
    t = re.sub(r"\s+", " ", URL_RE.sub(" ", t)).strip()
    if not any(c.isalpha() for c in t):
        return None

    return t


def prepare_comment(raw):
    t = clean_comment(raw)

    if t is None:
        return None, "bad"

    w = t.split()

    if len(w) < MIN_COMMENT_WORDS:
        return None, "short"

    # 1. Le sigle esplicite hanno priorità:
    #    ENTJ, INFP, ecc. -> leaky
    if TYPE_RE.search(t):
        if DROP_LEAKY_COMMENTS:
            return None, "leaky"

    # 2. Altri riferimenti espliciti rilevati dalle regex,
    #    esclusi i placeholder
    other_leak_patterns = (
        FUNC_RE,
        VOCAB_RE,
    )

    if DROP_LEAKY_COMMENTS and any(
        pattern.search(t) for pattern in other_leak_patterns
    ):
        return None, "leaky"

    # 3. I placeholder hanno uno stato separato
    if PLACEHOLDER_RE.search(t):
        return " ".join(w[:MAX_COMMENT_WORDS]), "with placeholder"

    # 4. Commento normale
    return " ".join(w[:MAX_COMMENT_WORDS]), "ok"


This code runs the cleaning and masking functions on all three datasets (Big Five, MBTI, and the final dataset).

In [ ]:

import os
import gc
import re
import html
import pandas as pd
from collections import Counter

try:
    import torch
except ImportError:
    torch = None


# ============================================================
# 1. PERCORSI DEI TRE DATASET ORIGINALI
# ============================================================

dataset_paths = [
    "/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/comments_final.csv",
    "/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/comments_mbti.csv",
    "/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/comments_big_five.csv",
]

# Riduci a 10_000 o 20_000 se hai poca RAM
CHUNK_SIZE = 20_000

MIN_COMMENT_WORDS = 5
MAX_COMMENT_WORDS = 500
DROP_LEAKY_COMMENTS = True

REPLACEMENTS = [
    (TYPE_RE, "TYPE_MENTION"),
    (FUNC_RE, "COGFUNC_MENTION"),
]


# ============================================================
# 2. SOSTITUZIONE DEI RIFERIMENTI ESPLICITI
# ============================================================

def replace_leaks(text):
    for pattern, placeholder in REPLACEMENTS:
        text = pattern.sub(placeholder, text)

    # Gli altri termini espliciti vengono sostituiti
    text = VOCAB_RE.sub("TYPE_MENTION", text)

    return text


# ============================================================
# 3. PULIZIA DI UN SINGOLO COMMENTO
# ============================================================

def clean_for_dataset(raw):
    cleaned, status = prepare_comment(raw)

    if status in ("bad", "short"):
        return None, status

    if status == "leaky":
        text = clean_comment(raw)

        if text is None:
            return None, "bad"

        text = replace_leaks(text)
        return text, "leaky"

    # Mantiene il testo pulito, compresi i placeholder esistenti
    return cleaned, status


# ============================================================
# 4. CICLO PRINCIPALE: UN DATASET ALLA VOLTA
# ============================================================

for dataset_number, path in enumerate(dataset_paths, start=1):

    print("\n" + "=" * 65)
    print(f"DATASET {dataset_number}/{len(dataset_paths)}")
    print(path)
    print("=" * 65)

    if not os.path.isfile(path):
        print(f"ERRORE: file non trovato: {path}")
        continue

    # File temporaneo nella stessa cartella del CSV originale.
    # L'originale rimane intatto fino al completamento.
    temp_path = path + ".cleaning_tmp"

    stats = Counter()
    total_input = 0
    total_output = 0

    try:
        # Legge soltanto l'intestazione per individuare la colonna
        header = pd.read_csv(path, nrows=0)
        columns = list(header.columns)
        del header
        gc.collect()

        comment_col = next(
            (c for c in ["body", "comment", "text"] if c in columns),
            columns[2] if len(columns) > 2 else None
        )

        if comment_col is None:
            raise ValueError(
                "Impossibile individuare la colonna dei commenti."
            )

        print(f"Colonna dei commenti: {comment_col}")
        print(f"Dimensione blocco: {CHUNK_SIZE:,} righe")
        print("Avvio elaborazione...")

        # Scrive il nuovo CSV a blocchi, senza caricare tutto in RAM
        first_chunk = True

        for chunk in pd.read_csv(
            path,
            chunksize=CHUNK_SIZE,
            low_memory=True
        ):
            total_input += len(chunk)

            # Costruisce soltanto il blocco pulito corrente
            cleaned_texts = []
            keep_positions = []

            for position, raw in enumerate(chunk[comment_col].values):
                cleaned, status = clean_for_dataset(raw)
                stats[status] += 1

                if status in ("bad", "short") or cleaned is None:
                    continue

                cleaned_texts.append(cleaned)
                keep_positions.append(position)

            # Conserva tutte le colonne delle righe valide
            cleaned_chunk = chunk.iloc[keep_positions].copy()

            # Sovrascrive SOLO la colonna dei commenti
            cleaned_chunk[comment_col] = cleaned_texts

            total_output += len(cleaned_chunk)

            # Scrive il blocco sul file temporaneo
            cleaned_chunk.to_csv(
                temp_path,
                mode="w" if first_chunk else "a",
                header=first_chunk,
                index=False,
                encoding="utf-8"
            )

            first_chunk = False

            del chunk, cleaned_chunk
            del cleaned_texts, keep_positions
            gc.collect()

            if torch is not None and torch.cuda.is_available():
                torch.cuda.empty_cache()

            print(
                f"Elaborati {total_input:,} commenti | "
                f"Mantenuti {total_output:,}",
                end="\r"
            )

        # Gestisce anche un CSV senza righe di dati
        if first_chunk:
            pd.DataFrame(columns=columns).to_csv(
                temp_path,
                index=False,
                encoding="utf-8"
            )

        # Sostituisce l'originale solo se l'elaborazione è terminata
        # correttamente.
        os.replace(temp_path, path)

        print("\n\nDataset salvato e sovrascritto correttamente.")
        print(f"Righe iniziali: {total_input:,}")
        print(f"Righe finali:   {total_output:,}")
        print(f"Righe rimosse:  {total_input - total_output:,}")

        print("\nStatistiche:")
        for status, count in sorted(stats.items()):
            percentage = (
                count / total_input * 100 if total_input else 0
            )
            print(
                f"  {status.upper()}: {count:,} "
                f"({percentage:.2f}%)"
            )

    except Exception as e:
        print(f"\nERRORE durante la pulizia: {e}")
        print("Il file originale non è stato sostituito "
              "se l'errore è avvenuto prima del salvataggio finale.")

        # Rimuove il temporaneo incompleto, se presente
        if os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except OSError:
                print(f"Attenzione: temporaneo da controllare: {temp_path}")

    finally:
        # Pulizia prima del dataset successivo
        gc.collect()

        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("Memoria temporanea liberata.")

print("\n" + "=" * 65)
print("CICLO COMPLETATO")
print("=" * 65)



DATASET 1/3
/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/comments_final.csv
Colonna dei commenti: body
Dimensione blocco: 20,000 righe
Avvio elaborazione...
Elaborati 895,742 commenti | Mantenuti 895,725

Dataset salvato e sovrascritto correttamente.
Righe iniziali: 895,742
Righe finali:   895,725
Righe rimosse:  17

Statistiche:
  BAD: 4 (0.00%)
  OK: 817,526 (91.27%)
  SHORT: 13 (0.00%)
  WITH PLACEHOLDER: 78,199 (8.73%)
Memoria temporanea liberata.

DATASET 2/3
/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/comments_mbti.csv
Colonna dei commenti: body
Dimensione blocco: 20,000 righe
Avvio elaborazione...


Dataset salvato e sovrascritto correttamente.
Righe iniziali: 12,539,934
Righe finali:   12,539,692
Righe rimosse:  242

Statistiche:
  BAD: 190 (0.00%)
  OK: 12,234,517 (97.56%)
  SHORT: 52 (0.00%)
  WITH PLACEHOLDER: 305,175 (2.43%)
Memoria temporanea liberata.

DATASET 3/3
/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/comments_big_five.csv
Colonna dei c

A questo punto, effettuiamo il MERGE con il dataset degli autori e creiamo i tre dataset contenenti gli autori relativi al Big Five, gli autori relativi all'MBTI, e gli autori del dataset finale.

Caricamento del file degli autori:

In [ ]:
author_profiles_path = '/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/author_profiles.csv'

try:
    df_authors = pd.read_csv(author_profiles_path)
    print('\nauthor_profiles.csv loaded successfully to GPU.')
    display(df_authors.head(1))
except FileNotFoundError:
    print(f"Error: '{author_profiles_path}' not found. Please ensure the file is in your Google Drive and you have correctly mounted it.")
except Exception as e:
    print(f"An error occurred while loading '{author_profiles_path}': {e}.")


mbti_columns = ["mbti", "introverted", "intuitive", "thinking", "perceiving"]
big_five_columns = ["agreeableness", "openness", "conscientiousness", "extraversion", "neuroticism"]
all_columns = mbti_columns + big_five_columns

# Final dataset
final_dataset = df_authors.dropna(subset=all_columns)
print(f"Shape of 'final_dataset': {final_dataset.shape}")
print("Head of 'final_dataset':")
display(final_dataset.head(1))

# MBTI dataset
mbti_dataset = df_authors.dropna(subset=mbti_columns)
mbti_dataset = mbti_dataset[~mbti_dataset['author'].isin(final_dataset['author'])]
print(f"Shape of 'mbti_dataset': {mbti_dataset.shape}")
print("Head of 'mbti_dataset':")
display(mbti_dataset.head(1))

# Big Five dataset
big_five_dataset = df_authors.dropna(subset=big_five_columns)
big_five_dataset = big_five_dataset[~big_five_dataset['author'].isin(final_dataset['author'])]
print(f"Shape of 'big_five_dataset': {big_five_dataset.shape}")
print("Head of 'big_five_dataset':")
display(big_five_dataset.head(1))

#Cleaning
mbti_columns = ["mbti", "introverted", "intuitive", "thinking", "perceiving"]
big_five_columns = ["agreeableness", "openness", "conscientiousness", "extraversion", "neuroticism"]
#mbti_dataset = mbti_dataset.drop(columns=big_five_columns)
#big_five_dataset = big_five_dataset.drop(columns=mbti_columns)

#Counting rows: was the partitioning successful?
len_mbti = len(mbti_dataset)
len_b5 = len(big_five_dataset)
len_final = len(final_dataset)
total_split = len_mbti + len_b5 + len_final

cond_mbti = df_authors[mbti_columns].notna().all(axis=1)
cond_b5 = df_authors[big_five_columns].notna().all(axis=1)
len_initial_dataset = len(df_authors[cond_mbti | cond_b5])

print(f"Authors in 'mbti_dataset': {len_mbti}")
print(f"Authors in 'big_five_dataset': {len_b5}")
print(f"Authors in 'final_dataset': {len_final}")
print(f"Sum of the authors in the three datasets: {total_split}")
print(f"Valid authors in the starting dataset: {len_initial_dataset}")
print(f"Verifica: {total_split} == {len_initial_dataset} -> {total_split == len_initial_dataset}")


author_profiles.csv loaded successfully to GPU.


,author,mbti,introverted,intuitive,thinking,perceiving,gender,age,enneagram,country,...,enneagram_wing,is_native_english_country,predicted_test,test_name,test_scale,16pers_ta,test_result_type,is_female,is_female_pred,is_female_proba
0,-Afrodisiac-,intp,1.0,1.0,1.0,1.0,m,19.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,0.531728


Shape of 'final_dataset': (377, 38)
Head of 'final_dataset':


,author,mbti,introverted,intuitive,thinking,perceiving,gender,age,enneagram,country,...,enneagram_wing,is_native_english_country,predicted_test,test_name,test_scale,16pers_ta,test_result_type,is_female,is_female_pred,is_female_proba
7,-BlitzN9ne,entp,0.0,1.0,1.0,1.0,m,21.0,3w4,NaN,...,4.0,NaN,1.0,qualtrics,percentiles,NaN,description,0.0,0,0.027725


Shape of 'mbti_dataset': (8690, 38)
Head of 'mbti_dataset':


,author,mbti,introverted,intuitive,thinking,perceiving,gender,age,enneagram,country,...,enneagram_wing,is_native_english_country,predicted_test,test_name,test_scale,16pers_ta,test_result_type,is_female,is_female_pred,is_female_proba
0,-Afrodisiac-,intp,1.0,1.0,1.0,1.0,m,19.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,0.531728


Shape of 'big_five_dataset': (1191, 38)
Head of 'big_five_dataset':


,author,mbti,introverted,intuitive,thinking,perceiving,gender,age,enneagram,country,...,enneagram_wing,is_native_english_country,predicted_test,test_name,test_scale,16pers_ta,test_result_type,is_female,is_female_pred,is_female_proba
1,-Areopagan-,NaN,NaN,NaN,NaN,NaN,m,NaN,NaN,NaN,...,NaN,NaN,0.0,understand myself,percentiles,NaN,percentiles,0.0,0,0.162758


Authors in 'mbti_dataset': 8690
Authors in 'big_five_dataset': 1191
Authors in 'final_dataset': 377
Sum of the authors in the three datasets: 10258
Valid authors in the starting dataset: 10258
Verifica: 10258 == 10258 -> True


Finally,et's merge these datasets with the authors' profiles:

In [ ]:

import os
import gc
import json
import sqlite3
import tempfile
import pandas as pd

# --------------------------------------------------
# 1. Percorsi
# --------------------------------------------------

base_path = (
    '/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets'
)

# Ordine importante: final -> Big Five -> MBTI
jobs = [
    {
        "variable": "final_dataset",
        "comments": os.path.join(base_path, "comments_final.csv"),
        "output": os.path.join(base_path, "final_complete_dataset.csv"),
        "batch_size": 300,
    },
    {
        "variable": "big_five_dataset",
        "comments": os.path.join(base_path, "comments_big_five.csv"),
        "output": os.path.join(base_path, "big_five_complete_dataset.csv"),
        "batch_size": 300,
    },
    {
        "variable": "mbti_dataset",
        "comments": os.path.join(base_path, "comments_mbti.csv"),
        "output": os.path.join(base_path, "mbti_complete_dataset.csv"),
        "batch_size": 300,
    },
]

CHUNK_SIZE = 20_000


# --------------------------------------------------
# 2. Funzione per liberare memoria GPU, se disponibile
# --------------------------------------------------

def clear_memory():
    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except (ImportError, RuntimeError):
        pass

    try:
        import cupy
        cupy.get_default_memory_pool().free_all_blocks()
        cupy.get_default_pinned_memory_pool().free_all_blocks()
    except (ImportError, RuntimeError):
        pass


# --------------------------------------------------
# 3. Elaborazione di una partizione alla volta
# --------------------------------------------------

for job in jobs:

    variable_name = job["variable"]
    comments_path = job["comments"]
    output_path = job["output"]
    batch_size = job["batch_size"]

    print("\n" + "=" * 65)
    print(f"PARTIZIONE: {variable_name}")
    print(f"Commenti:   {comments_path}")
    print(f"Output:     {output_path}")
    print("=" * 65)

    if variable_name not in globals():
        raise NameError(
            f"La variabile '{variable_name}' non esiste. "
            "Esegui prima la cella di creazione delle partizioni."
        )

    if not os.path.exists(comments_path):
        raise FileNotFoundError(
            f"File dei commenti non trovato: {comments_path}"
        )

    # Converte la partizione in Pandas, se proviene da cuDF.
    source_dataset = globals()[variable_name]

    if hasattr(source_dataset, "to_pandas"):
        authors_df = source_dataset.to_pandas()
    else:
        authors_df = source_dataset.copy()

    del source_dataset
    clear_memory()

    if "author" not in authors_df.columns:
        raise KeyError(
            f"La partizione '{variable_name}' non contiene la colonna 'author'."
        )

    authors_df["author"] = authors_df["author"].astype("string")

    # Gli autori da cercare nei commenti.
    author_values = authors_df["author"].dropna().unique().tolist()
    author_set = set(author_values)

    print(f"Autori nella partizione: {len(authors_df):,}")
    print(f"Autori distinti validi:  {len(author_set):,}")

    # Database temporaneo su disco, separato per ogni partizione.
    fd, sqlite_path = tempfile.mkstemp(
        prefix="comments_merge_",
        suffix=".sqlite",
        dir="/content"
    )
    os.close(fd)

    output_tmp = output_path + ".tmp"

    conn = None

    try:
        # ------------------------------------------
        # 4. Legge i commenti in blocchi e li salva
        #    su SQLite, mantenendo solo gli autori
        #    presenti nella partizione corrente.
        # ------------------------------------------

        conn = sqlite3.connect(sqlite_path)

        conn.execute("""
            CREATE TABLE comments (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                author TEXT NOT NULL,
                body TEXT,
                subreddit TEXT
            )
        """)

        print("\nImportazione dei commenti in blocchi...")

        imported_comments = 0

        for chunk in pd.read_csv(
            comments_path,
            chunksize=CHUNK_SIZE,
            low_memory=True,
            dtype={"author": "string"}
        ):

            required_columns = {"author", "body", "subreddit"}
            missing = required_columns - set(chunk.columns)

            if missing:
                raise KeyError(
                    f"Nel file {comments_path} mancano le colonne: {missing}"
                )

            # Conserva soltanto i commenti degli autori della partizione.
            mask = chunk["author"].isin(author_set)

            if mask.any():
                selected = chunk.loc[
                    mask, ["author", "body", "subreddit"]
                ].copy()

                selected["body"] = (
                    selected["body"].fillna("").astype(str)
                )
                selected["subreddit"] = (
                    selected["subreddit"].fillna("").astype(str)
                )

                records = (
                    (str(author), body, subreddit)
                    for author, body, subreddit
                    in selected.itertuples(index=False, name=None)
                    if pd.notna(author)
                )

                conn.executemany(
                    """
                    INSERT INTO comments (author, body, subreddit)
                    VALUES (?, ?, ?)
                    """,
                    records
                )

                imported_comments += len(selected)

                del selected, records

            del chunk, mask
            clear_memory()

        conn.commit()

        print(f"Commenti importati: {imported_comments:,}")

        # L'indice rende più veloci le ricerche per autore.
        conn.execute(
            "CREATE INDEX idx_comments_author ON comments(author)"
        )
        conn.commit()

        # ------------------------------------------
        # 5. Merge a batch di autori e salvataggio
        #    progressivo del risultato su CSV.
        # ------------------------------------------

        print("\nMerge e salvataggio a batch...")

        if os.path.exists(output_tmp):
            os.remove(output_tmp)

        total_batches = (
            len(authors_df) + batch_size - 1
        ) // batch_size

        for batch_number, start in enumerate(
            range(0, len(authors_df), batch_size), start=1
        ):

            author_batch = authors_df.iloc[
                start:start + batch_size
            ].copy()

            batch_authors = (
                author_batch["author"].dropna().unique().tolist()
            )

            aggregated = {}

            # SQLite limita il numero di parametri SQL:
            # batch_size=300 rimane sotto il limite comune di 999.
            if batch_authors:
                placeholders = ",".join(
                    "?" for _ in batch_authors
                )

                query = f"""
                    SELECT author, body, subreddit
                    FROM comments
                    WHERE author IN ({placeholders})
                    ORDER BY author, id
                """

                cursor = conn.execute(query, batch_authors)

                for author, body, subreddit in cursor:
                    if author not in aggregated:
                        aggregated[author] = {
                            "comments": [],
                            "subreddits": []
                        }

                    aggregated[author]["comments"].append(
                        body if body is not None else ""
                    )
                    aggregated[author]["subreddits"].append(
                        subreddit if subreddit is not None else ""
                    )

            # Associa a ogni autore le liste JSON.
            author_batch["comments"] = author_batch["author"].map(
                lambda author: json.dumps(
                    aggregated.get(
                        author,
                        {"comments": [], "subreddits": []}
                    )["comments"],
                    ensure_ascii=False
                )
            )

            author_batch["subreddits"] = author_batch["author"].map(
                lambda author: json.dumps(
                    aggregated.get(
                        author,
                        {"comments": [], "subreddits": []}
                    )["subreddits"],
                    ensure_ascii=False
                )
            )

            author_batch.to_csv(
                output_tmp,
                index=False,
                mode="w" if batch_number == 1 else "a",
                header=(batch_number == 1),
                encoding="utf-8"
            )

            print(
                f"Batch {batch_number}/{total_batches} completato "
                f"({len(author_batch):,} autori)"
            )

            del author_batch, aggregated, batch_authors
            clear_memory()

        # Se la partizione è vuota, crea comunque il CSV con le intestazioni.
        if len(authors_df) == 0:
            empty_df = authors_df.copy()
            empty_df["comments"] = pd.Series(dtype="object")
            empty_df["subreddits"] = pd.Series(dtype="object")
            empty_df.to_csv(
                output_tmp, index=False, encoding="utf-8"
            )
            del empty_df

        # ------------------------------------------
        # 6. Sostituisce l'output soltanto al termine
        #    dell'elaborazione.
        # ------------------------------------------

        conn.close()
        conn = None

        os.replace(output_tmp, output_path)

        print(f"\nCompletato: {output_path}")
        print(f"Autori salvati: {len(authors_df):,}")

        del authors_df

        # Libera anche la partizione originale, che potrebbe
        # essere un DataFrame cuDF residente sulla GPU.
        globals()[variable_name] = None

        clear_memory()

    except Exception:
        # Non lascia un CSV parziale al posto del risultato.
        if conn is not None:
            conn.close()
            conn = None

        if os.path.exists(output_tmp):
            os.remove(output_tmp)

        clear_memory()
        raise

    finally:
        if conn is not None:
            conn.close()

        if os.path.exists(sqlite_path):
            os.remove(sqlite_path)

        clear_memory()

    print("Memoria temporanea liberata.")


print("\n" + "=" * 65)
print("TUTTE LE PARTIZIONI SONO STATE ELABORATE.")
print("=" * 65)

print("\nFile creati:")
for job in jobs:
    print(job["output"])



PARTIZIONE: final_dataset
Commenti:   /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/comments_final.csv
Output:     /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/final_complete_dataset.csv
Autori nella partizione: 377
Autori distinti validi:  377

Importazione dei commenti in blocchi...
Commenti importati: 895,725

Merge e salvataggio a batch...
Batch 1/2 completato (300 autori)
Batch 2/2 completato (77 autori)

Completato: /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/final_complete_dataset.csv
Autori salvati: 377
Memoria temporanea liberata.

PARTIZIONE: big_five_dataset
Commenti:   /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/comments_big_five.csv
Output:     /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/big_five_complete_dataset.csv
Autori nella partizione: 1,191
Autori distinti validi:  1,191

Importazione dei commenti in blocchi...
Commenti importati: 1,683,442

Merge e salvataggio a batch...
Batch 1/4 completato (300 autori)
Batch 2/4 com

In [ ]:
import os

def search_for_file(filename, search_path):
    print(f"Scanning {search_path}...")
    matches = []
    for root, dirs, files in os.walk(search_path):
        for file in files:
            if filename.lower() in file.lower():
                full_path = os.path.join(root, file)
                matches.append(full_path)
                print(f"Found Match: {full_path}")
    return matches

results = search_for_file("mbti_complete_dataset.csv", "/content/drive/MyDrive")
if not results:
    print("\nNo exact file matches found. Let's list the top-level directories in your Drive to locate the 'Thesis_Elena_Maggiore' directory:")
    if os.path.exists("/content/drive/MyDrive"):
        print(os.listdir("/content/drive/MyDrive"))

Scanning /content/drive/MyDrive...
Found Match: /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/mbti_complete_dataset.csv


In [ ]:
import os
import gc
import json
import sqlite3
import tempfile
import pandas as pd

# 1. Clear Memory
def clear_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except (ImportError, RuntimeError):
        pass
    try:
        import cupy
        cupy.get_default_memory_pool().free_all_blocks()
        cupy.get_default_pinned_memory_pool().free_all_blocks()
    except (ImportError, RuntimeError):
        pass

clear_memory()
print("Memory cleared successfully.")

# 2. Paths and Parameters specifically for MBTI
base_path = '/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets'
mbti_job = {
    "variable": "mbti_dataset",
    "comments": os.path.join(base_path, "comments_mbti.csv"),
    "output": os.path.join(base_path, "mbti_complete_dataset.csv"),
    "batch_size": 300,
}

CHUNK_SIZE = 20_000
variable_name = mbti_job["variable"]
comments_path = mbti_job["comments"]
output_path = mbti_job["output"]
batch_size = mbti_job["batch_size"]

print("\n" + "=" * 65)
print(f"RE-RUNNING MERGE FOR: {variable_name}")
print(f"Commenti:   {comments_path}")
print(f"Output:     {output_path}")
print("=" * 65)

# Check if df_authors exists to recreate mbti_dataset dynamically if it was set to None
if 'mbti_dataset' not in globals() or globals()['mbti_dataset'] is None:
    print("Reconstructing 'mbti_dataset' from 'df_authors'...")
    if 'df_authors' not in globals():
        author_profiles_path = '/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/author_profiles.csv'
        df_authors = pd.read_csv(author_profiles_path)

    mbti_columns = ["mbti", "introverted", "intuitive", "thinking", "perceiving"]
    big_five_columns = ["agreeableness", "openness", "conscientiousness", "extraversion", "neuroticism"]
    all_columns = mbti_columns + big_five_columns

    final_df = df_authors.dropna(subset=all_columns)
    mbti_dataset = df_authors.dropna(subset=mbti_columns)
    mbti_dataset = mbti_dataset[~mbti_dataset['author'].isin(final_df['author'])]
else:
    mbti_dataset = globals()['mbti_dataset']

if hasattr(mbti_dataset, "to_pandas"):
    authors_df = mbti_dataset.to_pandas()
else:
    authors_df = mbti_dataset.copy()

clear_memory()

authors_df["author"] = authors_df["author"].astype("string")
author_values = authors_df["author"].dropna().unique().tolist()
author_set = set(author_values)

print(f"Autori nella partizione: {len(authors_df):,}")
print(f"Autori distinti validi:  {len(author_set):,}")

fd, sqlite_path = tempfile.mkstemp(
    prefix="comments_merge_mbti_",
    suffix=".sqlite",
    dir="/content"
)
os.close(fd)

output_tmp = output_path + ".tmp"
conn = None

try:
    conn = sqlite3.connect(sqlite_path)
    conn.execute("""
        CREATE TABLE comments (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            author TEXT NOT NULL,
            body TEXT,
            subreddit TEXT
        )
    """)

    print("\nImportazione dei commenti in blocchi...")
    imported_comments = 0

    for chunk in pd.read_csv(
        comments_path,
        chunksize=CHUNK_SIZE,
        low_memory=True,
        dtype={"author": "string"}
    ):
        mask = chunk["author"].isin(author_set)
        if mask.any():
            selected = chunk.loc[mask, ["author", "body", "subreddit"]].copy()
            selected["body"] = selected["body"].fillna("").astype(str)
            selected["subreddit"] = selected["subreddit"].fillna("").astype(str)

            records = (
                (str(author), body, subreddit)
                for author, body, subreddit
                in selected.itertuples(index=False, name=None)
                if pd.notna(author)
            )

            conn.executemany(
                """
                INSERT INTO comments (author, body, subreddit)
                VALUES (?, ?, ?)
                """,
                records
            )
            imported_comments += len(selected)
            del selected, records

        del chunk, mask
        clear_memory()

    conn.commit()
    print(f"Commenti importati: {imported_comments:,}")

    conn.execute("CREATE INDEX idx_comments_author ON comments(author)")
    conn.commit()

    print("\nMerge e salvataggio a batch...")
    if os.path.exists(output_tmp):
        os.remove(output_tmp)

    total_batches = (len(authors_df) + batch_size - 1) // batch_size

    for batch_number, start in enumerate(range(0, len(authors_df), batch_size), start=1):
        author_batch = authors_df.iloc[start:start + batch_size].copy()
        batch_authors = author_batch["author"].dropna().unique().tolist()
        aggregated = {}

        if batch_authors:
            placeholders = ",".join("?" for _ in batch_authors)
            query = f"""
                SELECT author, body, subreddit
                FROM comments
                WHERE author IN ({placeholders})
                ORDER BY author, id
            """
            cursor = conn.execute(query, batch_authors)
            for author, body, subreddit in cursor:
                if author not in aggregated:
                    aggregated[author] = {"comments": [], "subreddits": []}
                aggregated[author]["comments"].append(body if body is not None else "")
                aggregated[author]["subreddits"].append(subreddit if subreddit is not None else "")

        author_batch["comments"] = author_batch["author"].map(
            lambda author: json.dumps(
                aggregated.get(author, {"comments": [], "subreddits": []})["comments"],
                ensure_ascii=False
            )
        )

        author_batch["subreddits"] = author_batch["author"].map(
            lambda author: json.dumps(
                aggregated.get(author, {"comments": [], "subreddits": []})["subreddits"],
                ensure_ascii=False
            )
        )

        author_batch.to_csv(
            output_tmp,
            index=False,
            mode="w" if batch_number == 1 else "a",
            header=(batch_number == 1),
            encoding="utf-8"
        )
        print(f"Batch {batch_number}/{total_batches} completato ({len(author_batch):,} autori)", end="\r")
        del author_batch, aggregated, batch_authors
        clear_memory()

    conn.close()
    conn = None

    os.replace(output_tmp, output_path)
    print(f"\n\nCompletato con successo! File salvato in: {output_path}")

except Exception as e:
    if conn is not None:
        conn.close()
    if os.path.exists(output_tmp):
        os.remove(output_tmp)
    print(f"\nError occurred: {e}")

finally:
    if os.path.exists(sqlite_path):
        os.remove(sqlite_path)
    clear_memory()

Memory cleared successfully.

RE-RUNNING MERGE FOR: mbti_dataset
Commenti:   /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/comments_mbti.csv
Output:     /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/mbti_complete_dataset.csv
Autori nella partizione: 8,690
Autori distinti validi:  8,690

Importazione dei commenti in blocchi...
Commenti importati: 12,539,692

Merge e salvataggio a batch...


Completato con successo! File salvato in: /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets/mbti_complete_dataset.csv
